# Data overview

What the four BPD tables hold, how far they reach, and what to watch out for. This notebook answers no research question by itself; the other notebooks lean on it.

**Inputs:** `data/processed/calls_2021_2025.csv.gz`, `data/processed/crime_cases_2021_2025.csv.gz`, `data/processed/collisions.csv`, `data/processed/vehicle_stops.csv`

**Outputs:** `results/01_dataset_summary.csv`, `figures/01_monthly_records.png`, `results/01_time_checks.csv`

In [1]:
# Shared loaders, plot style, and paths
import pandas as pd

from bpdtools.load import load_calls, load_cases
from bpdtools.paths import repo_path
from bpdtools.plots import set_plot_style
from bpdtools.traffic import load_collisions, load_stops

set_plot_style()

# Result tables and figures go to fixed folders at the repository root
results = repo_path("results")
figures = repo_path("figures")
results.mkdir(exist_ok=True)
figures.mkdir(exist_ok=True)

# The processed tables, each through its shared loader. The key column is
# the one that makes a row unique, and the unit says what a row is.
tables = {
    "Calls for Service": (load_calls(), "Incident Number", "call"),
    "Crime cases": (load_cases(), "Case_Number", "crime case"),
    "Collisions": (load_collisions(), "ObjectId", "collision"),
    "Vehicle stops": (load_stops(), "LEA_Record_ID", "vehicle stop"),
}

## What is in each table

In [2]:
# One row per table: what a row is, how many rows, the date range, and
# how many rows have no local time
summary_rows = []
for name, (table, key, unit) in tables.items():
    # Each key must be unique, or a row would not be one event
    if not table[key].is_unique:
        raise ValueError(f"{name}: {key} repeats, so rows are not one {unit} each")
    times = table["local_time"]
    summary_rows.append(
        {
            "dataset": name,
            "one_row_is": unit,
            "key": key,
            "rows": len(table),
            "first_local_date": times.min().strftime("%Y-%m-%d"),
            "last_local_date": times.max().strftime("%Y-%m-%d"),
            "rows_without_time": int(times.isna().sum()),
        }
    )
summary = pd.DataFrame(summary_rows)
summary.to_csv(results / "01_dataset_summary.csv", index=False)
summary

,dataset,one_row_is,key,rows,first_local_date,last_local_date,rows_without_time
0,Calls for Service,call,Incident Number,307628,2021-01-01,2025-12-31,0
1,Crime cases,crime case,Case_Number,51418,2021-01-01,2025-12-31,0
2,Collisions,collision,ObjectId,16639,2007-12-03,2026-11-01,101
3,Vehicle stops,vehicle stop,LEA_Record_ID,7431,2024-01-01,2026-05-26,0
